# Running the QLD cleaning module
- cleans demand data from metropolitan Brisbane (QLD)
- processes it with the metadata
- saves cleaned data into /home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD

# Running the VIC cleaning module
- cleans demand data from metropolitan Brisbane (QLD)
- processes it with the metadata
- saves cleaned data into /home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD

In [ ]:
%pwd

%cd /home/565/pv3484/aus_substation_electricity/data/cleaning_data

In [ ]:
import sys
from pathlib import Path

# Tell Python where your VIC_cleaning.py module lives
sys.path.append("/home/565/pv3484/aus_substation_electricity/cleaning_data")

import VIC_cleaning as vic

# Absolute paths to your data
raw_vic_dir = Path("/home/565/pv3484/aus_substation_electricity/data/raw_data/VIC_demand")
metadata_path = Path("/home/565/pv3484/aus_substation_electricity/data/DNSP_Zone_Substation_Characteristics.csv")
output_root = Path("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC")

# Run cleaning
vic.save_cleaned_state_VIC(
    state_dir=raw_vic_dir,
    metadata_path=metadata_path,
    output_root=output_root,
    sigma=None,
    remove_constant=False,
    fill_small_gaps=False,
    max_gap=4,
    landuse_filters=None
)

print("VIC cleaning complete.")

## Looking at the new cleaned demand file
- looking to see if the data did get cleaned and the file can be used

In [ ]:
import pandas as pd
from pathlib import Path

project_root = Path("/home/565/pv3484/aus_substation_electricity")

vic_demand_path = project_root / "data" / "cleaned_data" / "VIC" / "demand.csv"
vic_meta_path   = project_root / "data" / "cleaned_data" / "VIC" / "metadata.csv"

demand_vic = pd.read_csv(vic_demand_path, index_col=0, parse_dates=True)
meta_vic   = pd.read_csv(vic_meta_path)

demand_vic.shape[1]

## Creating basic plot of all substation demand data on Aus Day for every year
- testing to see if the data is showing expected patterns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load cleaned VIC demand
vic_demand_path = Path("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC/demand.csv")
demand_vic = pd.read_csv(vic_demand_path, index_col=0, parse_dates=True)

# ---------------------------------------------------------
# Extract Australia Day for each year
# ---------------------------------------------------------

ausday_data = {}

for year in sorted(demand_vic.index.year.unique()):
    try:
        ausday_data[year] = demand_vic.loc[f"{year}-01-26"]
    except KeyError:
        # If a year doesn't have Jan 26 data, skip it
        continue

# ---------------------------------------------------------
# Select first 3 substations
# ---------------------------------------------------------

substations = demand_vic.columns[:1]

# ---------------------------------------------------------
# Create subplots
# ---------------------------------------------------------

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, sub in zip(axes, substations):
    for year, df in ausday_data.items():
        ax.plot(df.index, df[sub], label=str(year))

    ax.set_title(f"Substation: {sub}")
    ax.set_ylabel("Demand")

    # Hourly ticks for 24 hours
    hourly_ticks = pd.date_range(
        start=df.index.min().floor("D"),
        end=df.index.min().floor("D") + pd.Timedelta("23H"),
        freq="1H"
    )
    ax.set_xticks(hourly_ticks)

    ax.grid(True)

# ---------------------------------------------------------
# Format x-axis (only once because sharex=True)
# ---------------------------------------------------------

plt.xticks(rotation=45, ha="right")
axes[-1].set_xlabel("Time of Day")

# Add legend outside the last subplot
axes[-1].legend(title="Year", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()